In [ ]:
# 3. 경로 설정

from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "embedding":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "etl":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = (
    PROJECT_ROOT
    / "etl"
    / "embedding"
    / "전처리_데이터"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "etl"
    / "embedding"
    / "후처리_데이터"
)

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 32

SPLITS = {
    "train": "huggingface_epinfomax_mbti_korean_4axis_train.csv",
    "validation": "huggingface_epinfomax_mbti_korean_4axis_validation.csv",
    "test": "huggingface_epinfomax_mbti_korean_4axis_test.csv",
}

LABEL_COLUMNS = ["label", "mbti_type", "EI", "NS", "FT", "JP"]
TEXT_COLUMN = "text"

In [ ]:
# 4. 임베딩/저장 함수

import json
from datetime import datetime

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


def load_split_csv(split: str, filename: str) -> pd.DataFrame:
    path = INPUT_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")

    df = pd.read_csv(path, encoding="utf-8-sig")

    required_columns = [TEXT_COLUMN, *LABEL_COLUMNS]
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(f"{split} CSV missing columns: {missing}")

    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)

    return df


def encode_texts(
    model: SentenceTransformer,
    texts: list[str],
    split: str,
) -> np.ndarray:
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)

    if embeddings.ndim != 2:
        raise ValueError(f"{split} embeddings must be 2D. got {embeddings.shape}")

    if embeddings.shape[0] != len(texts):
        raise ValueError(
            f"{split} row mismatch: embeddings={embeddings.shape[0]}, texts={len(texts)}"
        )

    return embeddings


def save_split_outputs(
    split: str,
    df: pd.DataFrame,
    embeddings: np.ndarray,
) -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    embedding_path = OUTPUT_DIR / f"{split}_embeddings.npy"
    labels_path = OUTPUT_DIR / f"{split}_labels.csv"
    texts_path = OUTPUT_DIR / f"{split}_texts.csv"

    np.save(embedding_path, embeddings)

    df[LABEL_COLUMNS].to_csv(
        labels_path,
        index=False,
        encoding="utf-8-sig",
    )

    df[[TEXT_COLUMN]].to_csv(
        texts_path,
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "split": split,
        "rows": int(len(df)),
        "embedding_dim": int(embeddings.shape[1]),
        "embedding_shape": list(embeddings.shape),
        "embedding_file": str(embedding_path),
        "labels_file": str(labels_path),
        "texts_file": str(texts_path),
    }


def embed_split(
    model: SentenceTransformer,
    split: str,
    filename: str,
) -> dict:
    df = load_split_csv(split, filename)

    texts = df[TEXT_COLUMN].tolist()
    embeddings = encode_texts(model, texts, split)

    return save_split_outputs(split, df, embeddings)


def save_metadata(results: list[dict]) -> None:
    metadata = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "purpose": "Qwen3 embeddings for MBTI 4-axis ML training.",
        "embedding_model": MODEL_NAME,
        "embedding_backend": "sentence-transformers",
        "batch_size": BATCH_SIZE,
        "normalize_embeddings": True,
        "input_column": TEXT_COLUMN,
        "label_columns": LABEL_COLUMNS,
        "important_rule": "embeddings[i] must match labels.iloc[i] and texts.iloc[i]",
        "splits": results,
    }

    metadata_path = OUTPUT_DIR / "embedding_metadata.json"
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

In [ ]:
# 5. 모델 로드 및 실행

model = SentenceTransformer(
    MODEL_NAME,
    model_kwargs={"torch_dtype": "auto"},
)

results = []

for split, filename in SPLITS.items():
    print(f"\n=== Embedding {split} ===")
    result = embed_split(model, split, filename)
    results.append(result)

save_metadata(results)

print(json.dumps(
    {
        "status": "completed",
        "output_dir": str(OUTPUT_DIR),
        "splits": results,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
# 6. 결과 확인

import numpy as np
import pandas as pd

X_train = np.load(OUTPUT_DIR / "train_embeddings.npy")
y_train = pd.read_csv(OUTPUT_DIR / "train_labels.csv")

print(X_train.shape)
print(y_train.shape)
print(y_train.head())